# PhoBERT Drink Classifier — Pipeline Jupyter Notebook Complete

Notebook này triển khai hoàn chỉnh pipeline huấn luyện và đánh giá mô hình **PhoBERT (vinai/phobert-base-v2)** kết hợp **Multimodal Fusion MLP** để phân loại sản phẩm đồ uống từ hóa đơn bán lẻ.

### Danh mục Đồ uống chuẩn (11 loại theo `Product Categories.xlsx`):
1. **Bia**
2. **Rượu**
3. **Nước ngọt**
4. **Nước tăng lực**
5. **Nước khoáng/nước suối**
6. **Cà phê**
7. **Trà**
8. **Trà sữa**
9. **Sữa tươi**
10. **Sinh tố**
11. **Nước ép**

## Bước 1: Cài đặt các thư viện cần thiết

In [1]:
%pip install -q torch transformers datasets underthesea scikit-learn matplotlib seaborn pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


## Bước 2: Đọc Trực Tiếp Danh Mục từ `Product Categories.xlsx` & Khai báo Cấu hình
Mô hình đọc động file `Product Categories.xlsx` để lấy danh sách chính xác các loại sản phẩm (Bia, Rượu, Nước ngọt, Cà phê, Trà sữa...).

In [2]:
import os
import re
import sys
import time
import math
import random
import pickle
import unicodedata
import logging
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve, average_precision_score, f1_score
from transformers import AutoConfig, AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

# 1. Cố định seed
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Device đang sử dụng: {device}")

# 2. ĐỌC TRỰC TIẾP DANH MỤC TỪ FILE EXCEL Product Categories.xlsx
category_excel_paths = [
    "../6. Data/Product Categories.xlsx",
    "6. Data/Product Categories.xlsx",
    "6. Data/Product Categories.xlsx"
]
cat_path = None
for p in category_excel_paths:
    if os.path.exists(p):
        cat_path = p
        break

if cat_path:
    print(f"-> ĐỌC DANH MỤC TRỰC TIẾP TỪ FILE EXCEL: {cat_path}")
    df_cat = pd.read_excel(cat_path, sheet_name="Categories")
    df_cat.iloc[:, 0] = df_cat.iloc[:, 0].ffill()
    
    BEVERAGE_PRODUCTS = df_cat[df_cat.iloc[:, 0].astype(str).str.contains("Beverages|Đồ uống", case=False, na=False)].iloc[:, 1].dropna().unique().tolist()
    FOOD_PRODUCTS = df_cat[df_cat.iloc[:, 0].astype(str).str.contains("Food", case=False, na=False)].iloc[:, 1].dropna().unique().tolist()
    
    print(f"   [Excel] Đã nạp {len(BEVERAGE_PRODUCTS)} danh mục Đồ uống chuẩn từ Product Categories.xlsx:")
    for idx, p in enumerate(BEVERAGE_PRODUCTS, 1):
        print(f"     {idx}. {p}")
else:
    BEVERAGE_PRODUCTS = ["Bia", "Rượu", "Nước ngọt", "Nước tăng lực", "Nước khoáng/nước suối", "Cà phê", "Trà", "Trà sữa", "Sữa tươi", "Sinh tố", "Nước ép"]

DRINK_TYPO_DICT = {
    "nuoc co ga": "nước ngọt", "nuoc ngot": "nước ngọt", "ca phe": "cà phê",
    "cafe": "cà phê", "coffee": "cà phê", "cold brew": "cà phê cold brew", "coldbrew": "cà phê cold brew",
    "tra sua": "trà sữa", "milk tea": "trà sữa", "sua uong": "sữa tươi", "sua hat": "sữa tươi",
    "nuoc ep": "nước ép", "sinh to": "sinh tố", "nuoc loc": "nước khoáng/nước suối",
    "nuoc khoang": "nước khoáng/nước suối", "ruou vang": "rượu", "bia lon": "bia", "tang luc": "nước tăng lực",
}

@dataclass
class Config:
    model_name: str = "vinai/phobert-base-v2"
    max_length: int = 64
    backbone_lr: float = 1e-5
    head_lr: float = 5e-5
    weight_decay: float = 0.05
    batch_size: int = 16
    epochs: int = 6
    warmup_ratio: float = 0.1
    patience: int = 3
    use_amp: bool = True
    uncertainty_margin: float = 0.15

cfg = Config()

/Users/buidoanhaiyen/Documents/Vidimi/Code/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device đang sử dụng: mps


## Bước 3: Tiền xử lý văn bản tiếng Việt chuyên biệt (`VietnameseTextPreprocessor`)
Bao gồm: NFC Unicode, Lowercase, Regex loại bỏ đơn vị dung tích (`ml`, `l`, `kg`, `lon`, `chai`...), Mapping từ điển typo và Word Tokenize (`underthesea`).

In [3]:
class VietnameseTextPreprocessor:
    def __init__(self, typo_dict=None):
        self.typo_dict = typo_dict or DRINK_TYPO_DICT
        self.unit_re = re.compile(
            r"\s*[\(\[]?(\d+[\.,]?\d*\s*)?(kg|g|gr|gram|ml|l|lit|lít|lon|chai|gói|pack|hộp|túi|thùng|lốc|vỉ|miếng|cái|size\s*[smlxl]+)[\)\]]?\s*$",
            re.IGNORECASE
        )
        self.typo_items = sorted(self.typo_dict.items(), key=lambda x: len(x[0]), reverse=True)
        try:
            from underthesea import word_tokenize
            self.segmenter = lambda txt: word_tokenize(txt, format="text")
        except ImportError:
            self.segmenter = lambda txt: txt

    def __call__(self, text: str) -> str:
        if not isinstance(text, str) or not text.strip():
            return ""
        text = unicodedata.normalize("NFC", text).lower()
        text = self.unit_re.sub("", text).strip()
        for wrong, correct in self.typo_items:
            pattern = re.compile(r"(?<![\w])" + re.escape(wrong) + r"(?![\w])", re.IGNORECASE)
            text = pattern.sub(correct, text)
        text = re.sub(r"[^\w\s\-]", " ", text, flags=re.UNICODE)
        text = re.sub(r"\s+", " ", text).strip()
        text = self.segmenter(text)
        return text

preprocessor = VietnameseTextPreprocessor()
print("Test Preprocessor:", preprocessor("Trà sữa Matcha Macchiato 500ml lon"))

Test Preprocessor: trà sữa matcha macchiato 500 ml


## Bước 4: Đọc 100% Dữ Liệu Hóa Đơn Thực Tế từ `6. Data`
Sử dụng trực tiếp file dữ liệu hóa đơn thực tế `6. Data/Data Processed/bill_output_processed - brand.csv` (87,076 dòng từ WinMart, Bách Hóa Xanh, Highlands Coffee, Circle K...).

In [4]:
import os
import pandas as pd

target_filename = "bill_output_processed - brand.csv"
real_data_path = None

# Tìm kiếm file trong thư mục hiện tại và các thư mục cha/con
search_dirs = [".", "..", "../.."]
for d in search_dirs:
    for root, dirs, files in os.walk(d):
        if target_filename in files:
            real_data_path = os.path.join(root, target_filename)
            break
    if real_data_path:
        break

if not real_data_path:
    raise FileNotFoundError(f"Không tìm thấy file {target_filename} trong project!")

print(f"-> NẠP 100% DỮ LIỆU HÓA ĐƠN THỰC TẾ TỪ: {real_data_path}")
raw_df = pd.read_csv(real_data_path, usecols=["Item Name", "Item price clean", "Item Number Clean", "BrandName", "SectoLabel", "IndusLable"]).dropna(subset=["Item Name"])
raw_df = raw_df.rename(columns={"Item Name": "product_name", "Item price clean": "price", "Item Number Clean": "quantity"})
raw_df["price"] = pd.to_numeric(raw_df["price"], errors="coerce").fillna(15000.0)
raw_df["quantity"] = pd.to_numeric(raw_df["quantity"], errors="coerce").fillna(1.0)

# Gán nhãn tự động nhận diện đồ uống dựa trên SectorLabel/IndusLabel chuẩn và từ khóa mở rộng (bổ sung cold brew, macchiato, matcha, espresso...)
drink_sectors = {'1.1', '1.2', '1.3', '1.4', '1.5', '1.8', '1.10', '2.7', '4.10'}
excluded_sec = {'1.6', '1.7', '1.9'}

drink_kw = [
    'trà', 'tra', 'cà phê', 'ca phe', 'cafe', 'coffee', 'cold brew', 'coldbrew', 'espresso', 'americano',
    'latte', 'cappuccino', 'capuchino', 'macchiato', 'mocha', 'matcha', 'bạc xỉu', 'bac xiu', 'bac siu',
    'đá xay', 'da xay', 'g7', 'nescafe', 'passio', 'katinate', 'phúc long', 'phuc long', 'pepsi', 'coca',
    '7up', 'sprite', 'mirinda', 'fanta', 'soda', 'nước ngọt', 'có ga', 'schweppes', 'twister', 'sữa',
    'sua', 'milk', 'th true', 'vinamilk', 'dutch lady', 'milo', 'ovaltine', 'fami', 'yakult', 'nutriboost',
    'tăng lực', 'tang luc', 'red bull', 'redbull', 'sting', 'monster', 'number one', 'pocari', 'revive',
    'nước khoáng', 'nuoc khoang', 'nước lọc', 'nuoc loc', 'nước suối', 'nuoc suoi', 'lavie', 'aquafina',
    'dasani', 'evian', 'vĩnh hảo', 'nước ép', 'nuoc ep', 'juice', 'tropicana', 'vfresh', 'sinh tố',
    'sinh to', 'smoothie', 'bia', 'beer', 'tiger', 'heineken', '333', 'saigon', 'budweiser', 'strongbow',
    'rượu', 'ruou', 'wine', 'soju', 'vodka', 'whisky', 'sake', 'boba', 'kombucha', 'cacao', 'socola',
    'hồng trà', 'lục trà', 'nước'
]

def is_drink_row(row):
    sec = str(row.get('SectoLabel', '')).strip().replace('"', '')
    ind = str(row.get('IndusLable', '')).strip().replace('"', '')
    if sec in drink_sectors:
        return 1
    if ind == '1' and sec not in excluded_sec:
        return 1
    name = str(row.get('product_name', '')).lower()
    
    food_exclude = ['bánh', 'cơm', 'quẩy', 'xôi', 'kẹo', 'hạt', 'snack', 'bò', 'gà', 'heo', 'vịt', 'tôm', 'cua', 'cá', 'mực', 'ghẹ', 'ốc', 'nướng', 'chiên', 'xào', 'lẩu', 'mì', 'phở', 'bún', 'cháo', 'gỏi', 'nem', 'chả', 'khoai', 'bắp', 'ngô']
    drink_exceptions = ['cold brew', 'cà phê', 'trà', 'sữa', 'nước', 'pepsi', 'coca', 'sting', 'bò húc', 'red bull']
    
    if any(f in name for f in food_exclude) and not any(k in name for k in drink_exceptions):
        return 0
    if any(k in name for k in drink_kw):
        return 1
    return 0

raw_df["is_drink"] = raw_df.apply(is_drink_row, axis=1)
df = raw_df.sample(n=min(10000, len(raw_df)), random_state=42).reset_index(drop=True)

print(f"-> Đã nạp thành công {len(df):,} mẫu hóa đơn thực tế từ tổng số {len(raw_df):,} dòng!")
print("Phân phối nhãn dữ liệu thực tế:")
print(df["is_drink"].value_counts().rename({1: "Drink (Đồ uống)", 0: "Non-drink (Khác)"}))

-> NẠP 100% DỮ LIỆU HÓA ĐƠN THỰC TẾ TỪ: ../Data Processed/bill_output_processed - brand.csv
-> Đã nạp thành công 10,000 mẫu hóa đơn thực tế từ tổng số 86,965 dòng!
Phân phối nhãn dữ liệu thực tế:
is_drink
Non-drink (Khác)    6386
Drink (Đồ uống)     3614
Name: count, dtype: int64


## Bước 5: Dataset & DataLoader (Phân chia Stratified 70/15/15)

In [5]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

class DrinkDataset(Dataset):
    def __init__(self, texts, labels, num_feats, tokenizer, max_len=64):
        self.texts = texts
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.num_feats = torch.tensor(num_feats, dtype=torch.float32)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len, padding="max_length", truncation=True, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "num_feats": self.num_feats[idx],
            "labels": self.labels[idx],
        }

# Tiền xử lý text & scale đặc trưng số
df["proc_text"] = df["product_name"].apply(preprocessor)
prices = np.log1p(df["price"].values).reshape(-1, 1)
qtys = df["quantity"].values.reshape(-1, 1)
raw_num = np.hstack([prices, qtys])

# Split 70% Train / 15% Val / 15% Test
X_tv, X_test, y_tv, y_test, num_tv, num_test = train_test_split(df["proc_text"].tolist(), df["is_drink"].tolist(), raw_num, test_size=0.15, stratify=df["is_drink"], random_state=42)
X_tr, X_va, y_tr, y_va, num_tr, num_va = train_test_split(X_tv, y_tv, num_tv, test_size=0.1765, stratify=y_tv, random_state=42)

scaler = StandardScaler()
num_tr_scaled = scaler.fit_transform(num_tr)
num_va_scaled = scaler.transform(num_va)
num_te_scaled = scaler.transform(num_test)

train_loader = DataLoader(DrinkDataset(X_tr, y_tr, num_tr_scaled, tokenizer), batch_size=cfg.batch_size, shuffle=True)
val_loader   = DataLoader(DrinkDataset(X_va, y_va, num_va_scaled, tokenizer), batch_size=cfg.batch_size, shuffle=False)
test_loader  = DataLoader(DrinkDataset(X_test, y_test, num_te_scaled, tokenizer), batch_size=cfg.batch_size, shuffle=False)

print(f"Tập Train: {len(X_tr)} mẫu | Val: {len(X_va)} mẫu | Test: {len(X_test)} mẫu")

Tập Train: 6999 mẫu | Val: 1501 mẫu | Test: 1500 mẫu


## Bước 6: Khởi tạo Kiến trúc Multimodal PhoBERT + MLP

In [6]:
class NumericalMLP(nn.Module):
    def __init__(self, in_dim=2, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
    def forward(self, x):
        return self.net(x)

class MultimodalDrinkClassifier(nn.Module):
    def __init__(self, model_name=cfg.model_name):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name, attn_implementation="eager")
                # Đóng băng Embeddings và 4 lớp Encoder đầu tiên
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        for layer in self.bert.encoder.layer[:4]:
            for param in layer.parameters():
                param.requires_grad = False
        self.num_mlp = NumericalMLP(in_dim=2, hidden_dim=32)
        fused_dim = self.bert.config.hidden_size + 32 # 768 + 32 = 800
        self.fusion_head = nn.Sequential(
            nn.Linear(fused_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1),
        )
    def forward(self, input_ids, attention_mask, num_feats):
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb  = bert_out.pooler_output
        num_emb  = self.num_mlp(num_feats)
        fused    = torch.cat([cls_emb, num_emb], dim=-1)
        return self.fusion_head(fused).squeeze(-1)

model = MultimodalDrinkClassifier().to(device)
print("Khởi tạo mô hình Multimodal PhoBERT + MLP thành công!")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 32305.50it/s]
[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Khởi tạo mô hình Multimodal PhoBERT + MLP thành công!


## Bước 7: Huấn luyện Mô hình & Quét Ngưỡng Tối Ưu (Threshold Tuning)

In [7]:
import copy

# Differential Learning Rate setup
no_decay = ["bias", "LayerNorm.weight"]
optimizer_grouped_parameters = [
    {"params": [p for n, p in model.bert.named_parameters() if not any(nd in n for nd in no_decay)], "lr": cfg.backbone_lr, "weight_decay": cfg.weight_decay},
    {"params": [p for n, p in model.bert.named_parameters() if any(nd in n for nd in no_decay)], "lr": cfg.backbone_lr, "weight_decay": 0.0},
    {"params": [p for n, p in list(model.num_mlp.named_parameters()) + list(model.fusion_head.named_parameters()) if not any(nd in n for nd in no_decay)], "lr": cfg.head_lr, "weight_decay": cfg.weight_decay},
    {"params": [p for n, p in list(model.num_mlp.named_parameters()) + list(model.fusion_head.named_parameters()) if any(nd in n for nd in no_decay)], "lr": cfg.head_lr, "weight_decay": 0.0},
]
optimizer = torch.optim.AdamW(optimizer_grouped_parameters)

# Thêm Scheduler
total_steps = len(train_loader) * cfg.epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * cfg.warmup_ratio),
    num_training_steps=total_steps
)

# AMP (Mixed Precision) - Chỉ bật trên CUDA (Nvidia GPU) để tránh lỗi với MPS (Mac)
use_amp = cfg.use_amp and torch.cuda.is_available()
amp_scaler = torch.amp.GradScaler('cuda') if use_amp else None

# pos_weight tự động
n_pos = sum(y_tr)
n_neg = len(y_tr) - n_pos
pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print(f"Huấn luyện {cfg.epochs} epochs... pos_weight = {pos_weight.item():.2f}")

best_val_loss = float('inf')
patience_counter = 0
best_model_state = None

for epoch in range(cfg.epochs):
    model.train()
    total_train_loss = 0.0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attn_mask = batch["attention_mask"].to(device)
        num_feats = batch["num_feats"].to(device)
        labels    = batch["labels"].to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        if use_amp:
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits = model(input_ids, attn_mask, num_feats)
                loss   = criterion(logits, labels)
        else:
            logits = model(input_ids, attn_mask, num_feats)
            loss   = criterion(logits, labels)
            
        # Backward pass
        if amp_scaler:
            amp_scaler.scale(loss).backward()
            amp_scaler.step(optimizer)
            amp_scaler.update()
        else:
            loss.backward()
            optimizer.step()
            
        scheduler.step()
        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    # --- EVALUATION MỖI EPOCH ĐỂ LƯU MODEL VÀ EARLY STOPPING ---
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)
            num_feats = batch["num_feats"].to(device)
            labels    = batch["labels"].to(device)
            
            if use_amp:
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    logits = model(input_ids, attn_mask, num_feats)
                    loss = criterion(logits, labels)
            else:
                logits = model(input_ids, attn_mask, num_feats)
                loss = criterion(logits, labels)
                
            total_val_loss += loss.item()
            
    avg_val_loss = total_val_loss / len(val_loader)
    
    print(f"Epoch {epoch + 1}/{cfg.epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")
    
    # Model Checkpointing & Early Stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        best_model_state = copy.deepcopy(model.state_dict())
        torch.save(best_model_state, "best_phobert_drink_model.pth")
        print("  -> Đã lưu mô hình tốt nhất (Best Checkpoint)!")
    else:
        patience_counter += 1
        print(f"  -> Early Stopping Patience: {patience_counter}/{cfg.patience}")
        if patience_counter >= cfg.patience:
            print(f"Early stopping kích hoạt ở Epoch {epoch + 1}! Dừng huấn luyện để tránh Overfitting.")
            break

# Nạp lại weights tốt nhất trước khi dò Threshold
print("\nĐang tải lại tham số mô hình tốt nhất từ quá trình huấn luyện...")
if best_model_state:
    model.load_state_dict(best_model_state)

# Quét ngưỡng tối ưu trên Validation với model tốt nhất
model.eval()
val_logits, val_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        logits = model(batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["num_feats"].to(device))
        val_logits.extend(logits.cpu().numpy())
        val_labels.extend(batch["labels"].numpy())

val_probs = 1 / (1 + np.exp(-np.array(val_logits)))
best_thresh, best_f1 = 0.5, 0.0
for th in np.arange(0.1, 0.9, 0.05):
    f1 = f1_score(val_labels, (val_probs >= th).astype(int), zero_division=0)
    if f1 > best_f1:
        best_f1, best_thresh = f1, th

print(f"\nNgưỡng tối ưu tìm được trên tập Val (từ mô hình tốt nhất): {best_thresh:.2f} (Val F1 = {best_f1:.4f})")

Huấn luyện 6 epochs... pos_weight = 1.77
Epoch 1/6 | Train Loss: 0.6877 | Val Loss: 0.4189 | LR: 9.26e-06
  -> Đã lưu mô hình tốt nhất (Best Checkpoint)!
Epoch 2/6 | Train Loss: 0.4025 | Val Loss: 0.3488 | LR: 7.40e-06
  -> Đã lưu mô hình tốt nhất (Best Checkpoint)!
Epoch 3/6 | Train Loss: 0.3279 | Val Loss: 0.3259 | LR: 5.55e-06
  -> Đã lưu mô hình tốt nhất (Best Checkpoint)!
Epoch 4/6 | Train Loss: 0.2816 | Val Loss: 0.3178 | LR: 3.70e-06
  -> Đã lưu mô hình tốt nhất (Best Checkpoint)!
Epoch 5/6 | Train Loss: 0.2414 | Val Loss: 0.3072 | LR: 1.85e-06
  -> Đã lưu mô hình tốt nhất (Best Checkpoint)!
Epoch 6/6 | Train Loss: 0.2282 | Val Loss: 0.3057 | LR: 0.00e+00
  -> Đã lưu mô hình tốt nhất (Best Checkpoint)!

Đang tải lại tham số mô hình tốt nhất từ quá trình huấn luyện...

Ngưỡng tối ưu tìm được trên tập Val (từ mô hình tốt nhất): 0.80 (Val F1 = 0.8991)


## Bước 8: Đánh giá Độc lập trên Tập Test (Report + ROC-AUC)

In [8]:
model.eval()
test_logits, test_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        logits = model(batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["num_feats"].to(device))
        test_logits.extend(logits.cpu().numpy())
        test_labels.extend(batch["labels"].numpy())

test_probs = 1 / (1 + np.exp(-np.array(test_logits)))
test_preds = (test_probs >= best_thresh).astype(int)

print("=" * 60)
print(f"BÁO CÁO ĐÁNH GIÁ TẬP TEST (Threshold = {best_thresh:.2f})")
print("=" * 60)
print(classification_report(test_labels, test_preds, target_names=["Non-drink", "Drink"], digits=4))
print(f"ROC-AUC Score: {roc_auc_score(test_labels, test_probs):.4f}")

BÁO CÁO ĐÁNH GIÁ TẬP TEST (Threshold = 0.80)
              precision    recall  f1-score   support

   Non-drink     0.9176    0.9760    0.9459       958
       Drink     0.9522    0.8450    0.8954       542

    accuracy                         0.9287      1500
   macro avg     0.9349    0.9105    0.9206      1500
weighted avg     0.9301    0.9287    0.9276      1500

ROC-AUC Score: 0.9698


## Bước 9: Suy Luận (Inference) Trực Tiếp Trên Các Dòng Hóa Đơn Thực Tế
Không dùng dữ liệu mẫu giả lập nữa. Mô hình nạp trực tiếp các mẫu hóa đơn từ `bill_output_processed - brand.csv` và bóc tách loại món theo 11 danh mục chuẩn của `Product Categories.xlsx`.

In [11]:
def classify_drink_subclass(text, is_drink):
    t = text.lower()
    
    # Bước lọc cứng (Hard filter): Nếu chứa từ khóa đồ ăn và không chứa từ khóa đồ uống đặc biệt -> chắc chắn là đồ ăn
    food_exclude = ['bánh', 'cơm', 'quẩy', 'xôi', 'kẹo', 'hạt', 'snack', 'bò', 'gà', 'heo', 'vịt', 'tôm', 'cua', 'cá', 'mực', 'ghẹ', 'ốc', 'nướng', 'chiên', 'xào', 'lẩu', 'mì', 'phở', 'bún', 'cháo', 'gỏi', 'nem', 'chả', 'khoai', 'bắp', 'ngô']
    drink_exceptions = ['cold brew', 'cà phê', 'trà', 'sữa', 'nước', 'pepsi', 'coca', 'sting', 'bò húc', 'red bull']
    if any(f in t for f in food_exclude) and not any(k in t for k in drink_exceptions):
        return "Khác / Không phải đồ uống"

    if not is_drink:
        return "Khác / Không phải đồ uống"
        
    if any(k in t for k in ["trà sữa", "tra sua", "milk tea", "macchiato", "bubble tea", "tiger sugar"]):
        return "Trà sữa"
    if any(k in t for k in ["cà phê", "ca phe", "cafe", "coffee", "g7", "nescafe", "highlands", "cold brew", "latte", "espresso"]):
        return "Cà phê"
    if any(k in t for k in ["trà", "tra", "tea", "oolong", "nestea", "lipton", "trà xanh", "trà đào", "trà chanh"]):
        return "Trà"
    if any(k in t for k in ["sữa tươi", "sữa hạt", "milo", "ovaltine", "fami", "sữa đậu", "sua uong"]):
        return "Sữa tươi"
    if any(k in t for k in ["pepsi", "coca", "7up", "sprite", "mirinda", "fanta", "soda", "nước ngọt", "có ga"]):
        return "Nước ngọt"
    if any(k in t for k in ["tăng lực", "red bull", "sting", "monster", "number one", "pocari", "revive"]):
        return "Nước tăng lực"
    if any(k in t for k in ["nước ép", "juice", "tropicana", "vfresh", "nước cam", "nước dừa"]):
        return "Nước ép"
    if any(k in t for k in ["sinh tố", "smoothie"]):
        return "Sinh tố"
    if any(k in t for k in ["bia", "beer", "tiger", "heineken", "333", "saigon", "budweiser", "strongbow"]):
        return "Bia"
    if any(k in t for k in ["rượu", "wine", "soju", "vodka", "whisky", "sake"]):
        return "Rượu"
    if any(k in t for k in ["nước khoáng", "nước lọc", "lavie", "aquafina", "dasani", "evian", "vĩnh hảo", "nước suối"]):
        return "Nước khoáng/nước suối"
    return "Đồ uống khác"

# Nạp mẫu trực tiếp từ file hóa đơn thực tế (chỉ dùng tên file)
target_filename = "bill_output_processed - brand.csv"
real_infer_path = None

search_dirs = [".", "..", "../.."]
for d in search_dirs:
    for root, dirs, files in os.walk(d):
        if target_filename in files:
            real_infer_path = os.path.join(root, target_filename)
            break
    if real_infer_path:
        break

if real_infer_path:
    print(f"-> SUY LUẬN TRỰC TIẾP TRÊN CÁC DÒNG HÓA ĐƠN THỰC TẾ TỪ: {real_infer_path}")
    raw_infer_df = pd.read_csv(real_infer_path, usecols=["Item Name", "Item price clean", "Item Number Clean", "BrandName"]).dropna(subset=["Item Name"]).sample(20)
    
    test_items  = raw_infer_df["Item Name"].tolist()
    test_prices = pd.to_numeric(raw_infer_df["Item price clean"], errors="coerce").fillna(15000.0).tolist()
    test_qtys   = pd.to_numeric(raw_infer_df["Item Number Clean"], errors="coerce").fillna(1.0).tolist()
    test_brands = raw_infer_df["BrandName"].tolist()
    
    model.eval()
    results = []
    for text, price, qty, brand in zip(test_items, test_prices, test_qtys, test_brands):
        proc_t = preprocessor(text)
        enc = tokenizer(proc_t, max_length=cfg.max_length, padding="max_length", truncation=True, return_tensors="pt")
        raw_n = np.array([[np.log1p(price), qty]])
        num_n = torch.tensor(scaler.transform(raw_n), dtype=torch.float32).to(device)
        
        with torch.no_grad():
            logit = model(enc["input_ids"].to(device), enc["attention_mask"].to(device), num_n)
            prob  = float(torch.sigmoid(logit).cpu().item())
            
        is_drink = prob >= best_thresh
        subclass = classify_drink_subclass(text, is_drink)
        
        # Đồng bộ nhãn is_drink dựa trên subclass để fix các ca False Positive như "bò nướng tảng"
        if subclass == "Khác / Không phải đồ uống":
            is_drink = False
            
        flag_review = 1 if abs(prob - best_thresh) < cfg.uncertainty_margin else 0
        
        results.append({
            "raw_text": text,
            "brand": brand,
            "price": int(price),
            "prob": prob,
            "is_drink": "DRINK" if is_drink else "NON-DRINK",
            "category": subclass,
            "flag": "CAN REVIEW" if flag_review else "Tu dong"
        })
        
    res_df = pd.DataFrame(results)
    print("\n" + "─" * 105)
    print(f"{'Sản phẩm thực tế':<35} {'Cửa hàng/Brand':<18} {'Giá tiền':>9}  {'Prob':>6}  {'Kết quả':<10}  {'Loại món (11 nhãn)':<22}")
    print("─" * 105)
    for _, r in res_df.iterrows():
        print(f"{r['raw_text']:<35} {r['brand']:<18} {r['price']:>9,d}  {r['prob']:>6.3f}  {r['is_drink']:<10}  {r['category']:<22}")
    print("─" * 105)
else:
    print(f"Không tìm thấy file {target_filename}.")

-> SUY LUẬN TRỰC TIẾP TRÊN CÁC DÒNG HÓA ĐƠN THỰC TẾ TỪ: ../Data Processed/bill_output_processed - brand.csv

─────────────────────────────────────────────────────────────────────────────────────────────────────────
Sản phẩm thực tế                    Cửa hàng/Brand      Giá tiền    Prob  Kết quả     Loại món (11 nhãn)    
─────────────────────────────────────────────────────────────────────────────────────────────────────────
Pin sạc dự phòng PowerG Smart 12w 10000mAh Innostyle Đen Thế giới di động     599,000   0.049  NON-DRINK   Khác / Không phải đồ uống
SUA VAN TUC                         GO                    29,500   0.983  DRINK       Đồ uống khác          
Espresso                            Thương hiệu khác           4   0.950  DRINK       Cà phê                
NOW DEAL 50%                        Thương hiệu khác      62,500   0.049  NON-DRINK   Khác / Không phải đồ uống
chả cá odeng hàn quốc               Aeon Mall             14,700   0.028  NON-DRINK   Khác / Không phải đồ 

In [12]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
import torch
import numpy as np

# Đọc file test.xlsx (File nằm ở thư mục 6. Data, ngang hàng với Code)
test_file_path = "../test.xlsx"
df_test = pd.read_excel(test_file_path)
df_test = df_test.dropna(subset=["Product Name"])

print(f"Đã tải {len(df_test)} mẫu từ test.xlsx")

test_items = df_test["Product Name"].tolist()
# Cột Product Category là nhãn thủ công (ground truth)
true_categories = df_test["Product Category"].fillna("Khác / Không phải đồ uống").tolist()

# Xử lý mapping nhãn cho đồng nhất với 11 danh mục của mô hình
def map_to_standard(label):
    lbl = str(label).lower().strip()
    if lbl in ["trà sữa"]: return "Trà sữa"
    if lbl in ["cà phê", "cafe"]: return "Cà phê"
    if lbl in ["trà"]: return "Trà"
    if lbl in ["sữa tươi"]: return "Sữa tươi"
    if lbl in ["nước ngọt"]: return "Nước ngọt"
    if lbl in ["nước tăng lực"]: return "Nước tăng lực"
    if lbl in ["nước ép"]: return "Nước ép"
    if lbl in ["sinh tố"]: return "Sinh tố"
    if lbl in ["bia"]: return "Bia"
    if lbl in ["rượu"]: return "Rượu"
    if lbl in ["nước khoáng/nước suối", "nước khoáng", "nước lọc"]: return "Nước khoáng/nước suối"
    if lbl in ["đồ uống khác"]: return "Đồ uống khác"
    return "Khác / Không phải đồ uống"

y_true = [map_to_standard(lbl) for lbl in true_categories]

# Dự đoán
model.eval()
y_pred = []
test_prices = [15000.0] * len(test_items) # dummy price
test_qtys = [1.0] * len(test_items)       # dummy quantity

for text, price, qty in zip(test_items, test_prices, test_qtys):
    proc_t = preprocessor(text)
    enc = tokenizer(proc_t, max_length=cfg.max_length, padding="max_length", truncation=True, return_tensors="pt")
    
    raw_n = np.array([[np.log1p(price), qty]])
    num_n = torch.tensor(scaler.transform(raw_n), dtype=torch.float32).to(device)
    
    with torch.no_grad():
        logit = model(enc["input_ids"].to(device), enc["attention_mask"].to(device), num_n)
        prob  = float(torch.sigmoid(logit).cpu().item())
        
    is_drink = prob >= best_thresh
    subclass = classify_drink_subclass(text, is_drink)
    y_pred.append(subclass)

# Tính toán tỉ lệ đánh đúng
correct = sum(1 for true, pred in zip(y_true, y_pred) if true == pred)
total = len(y_true)
accuracy = correct / total * 100

print(f"\n--- KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST.XLSX ---")
print(f"Tổng số mẫu: {total}")
print(f"Dự đoán đúng: {correct}")
print(f"Tỉ lệ đánh đúng (Accuracy): {accuracy:.2f}%\n")
print(classification_report(y_true, y_pred, zero_division=0))

Đã tải 1602 mẫu từ test.xlsx

--- KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST.XLSX ---
Tổng số mẫu: 1602
Dự đoán đúng: 1104
Tỉ lệ đánh đúng (Accuracy): 68.91%

                           precision    recall  f1-score   support

                      Bia       0.00      0.00      0.00         1
                   Cà phê       0.82      0.46      0.59        61
Khác / Không phải đồ uống       0.97      0.70      0.81      1281
    Nước khoáng/nước suối       1.00      0.80      0.89        15
                Nước ngọt       1.00      0.79      0.88        38
            Nước tăng lực       0.33      0.50      0.40         2
                  Nước ép       1.00      1.00      1.00         3
                     Rượu       0.00      0.00      0.00         0
                  Sinh tố       1.00      0.17      0.29         6
                 Sữa tươi       1.00      0.19      0.32        52
                      Trà       0.92      0.84      0.88       122
                  Trà sữa       0.95      0.95  